# Usage & Cost Admin API Cookbook

**프로그램을 통해 Claude API 사용량 및 비용 데이터에 접근하는 실용 가이드**

### 가능한 작업

**사용량 추적:**
- Token 소비량 모니터링(cache되지 않은 input, output, cache 생성/읽기)
- 모델, workspace, API key별 사용량 추적
- Cache 효율 및 server tool 사용량 분석

**비용 분석:**
- 서비스 유형별 상세 비용 내역 조회
- Workspace별 지출 추세 모니터링
- 재무 및 비용 배부 시나리오를 위한 보고서 생성

**일반적인 사용 사례:**
- **사용량 모니터링**: 소비 패턴 추적 및 비용 최적화
- **비용 귀속**: Workspace별로 팀/프로젝트에 비용 배분
- **Cache 분석**: Cache 효율 측정 및 개선
- **재무 보고**: 경영진용 요약 및 예산 보고서 생성

### API 개요

두 가지 주요 endpoint:
1. **Messages Usage API**: 유연한 그룹화가 가능한 token 단위 사용량 데이터
2. **Cost API**: 서비스별 내역이 포함된 USD 재무 데이터

### 사전 준비 및 보안

- **Admin API Key**: [Claude Console](https://console.anthropic.com/settings/admin-keys)에서 발급(형식: `sk-ant-admin...`)
- **보안**: Key를 환경 변수에 저장하고 정기적으로 교체하며 version control에 절대 commit하지 않기

In [ ]:
import os
from datetime import datetime, time, timedelta
from typing import Any

import requests


class AnthropicAdminAPI:
    """Anthropic Admin API endpoint의 보안 wrapper."""

    def __init__(self, api_key: str | None = None):
        self.api_key = api_key or os.getenv("ANTHROPIC_ADMIN_API_KEY")
        if not self.api_key:
            raise ValueError(
                "Admin API key required. Set ANTHROPIC_ADMIN_API_KEY environment variable."
            )

        if not self.api_key.startswith("sk-ant-admin"):
            raise ValueError("Invalid Admin API key format.")

        self.base_url = "https://api.anthropic.com/v1/organizations"
        self.headers = {
            "anthropic-version": "2023-06-01",
            "x-api-key": self.api_key,
            "Content-Type": "application/json",
        }

    def _make_request(self, endpoint: str, params: dict[str, Any]) -> dict[str, Any]:
        """기본 오류 처리를 포함한 인증 요청을 보냅니다."""
        url = f"{self.base_url}/{endpoint}"

        try:
            response = requests.get(url, headers=self.headers, params=params, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.HTTPError as e:
            if response.status_code == 401:
                raise ValueError("Invalid API key or insufficient permissions") from e
            elif response.status_code == 429:
                raise requests.exceptions.RequestException(
                    "Rate limit exceeded - try again later"
                ) from e
            else:
                raise requests.exceptions.RequestException(f"API error: {e}") from e


# 연결 테스트
def test_connection():
    try:
        client = AnthropicAdminAPI()

        # 간단한 테스트 query - bucket 경계에 맞추어 하루 시작 시점으로 정렬
        params = {
            "starting_at": (
                datetime.combine(datetime.utcnow(), time.min) - timedelta(days=1)
            ).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "ending_at": datetime.combine(datetime.utcnow(), time.min).strftime(
                "%Y-%m-%dT%H:%M:%SZ"
            ),
            "bucket_width": "1d",
            "limit": 1,
        }

        client._make_request("usage_report/messages", params)
        print("✅ Connection successful!")
        return client

    except Exception as e:
        print(f"❌ Connection failed: {e}")
        return None


client = test_connection()

## 기본 사용량 및 비용 추적

### 사용량 데이터 이해하기

Messages Usage API는 집계된 사용량을 담은 고정 간격인 **time bucket**으로 token 소비량을 제공합니다.

**주요 지표:**
- **uncached_input_tokens**: 새로운 input token(prompt, system message)
- **output_tokens**: Claude의 응답
- **cache_creation**: 재사용을 위해 cache된 token
- **cache_read_input_tokens**: 이전에 cache되어 재사용된 token

### 기본 사용량 조회

In [12]:
def get_daily_usage(client, days_back=7):
    """최근 N일의 사용량 데이터를 가져옵니다."""
    end_time = datetime.combine(datetime.utcnow(), time.min)
    start_time = end_time - timedelta(days=days_back)

    params = {
        "starting_at": start_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "ending_at": end_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "bucket_width": "1d",
        "limit": days_back,
    }

    return client._make_request("usage_report/messages", params)


def analyze_usage_data(response):
    """사용량 데이터를 처리하고 표시합니다."""
    if not response or not response.get("data"):
        print("No usage data found.")
        return

    total_uncached_input = total_output = total_cache_creation = 0
    total_cache_reads = total_web_searches = 0
    daily_data = []

    for bucket in response["data"]:
        date = bucket["starting_at"][:10]

        # Bucket의 모든 결과 합산
        bucket_uncached = bucket_output = bucket_cache_creation = 0
        bucket_cache_reads = bucket_web_searches = 0

        for result in bucket["results"]:
            bucket_uncached += result.get("uncached_input_tokens", 0)
            bucket_output += result.get("output_tokens", 0)

            cache_creation = result.get("cache_creation", {})
            bucket_cache_creation += cache_creation.get(
                "ephemeral_1h_input_tokens", 0
            ) + cache_creation.get("ephemeral_5m_input_tokens", 0)
            bucket_cache_reads += result.get("cache_read_input_tokens", 0)

            server_tools = result.get("server_tool_use", {})
            bucket_web_searches += server_tools.get("web_search_requests", 0)

        daily_data.append(
            {
                "date": date,
                "uncached_input_tokens": bucket_uncached,
                "output_tokens": bucket_output,
                "cache_creation": bucket_cache_creation,
                "cache_reads": bucket_cache_reads,
                "web_searches": bucket_web_searches,
                "total_tokens": bucket_uncached + bucket_output,
            }
        )

        # 합계에 추가
        total_uncached_input += bucket_uncached
        total_output += bucket_output
        total_cache_creation += bucket_cache_creation
        total_cache_reads += bucket_cache_reads
        total_web_searches += bucket_web_searches

    # Cache 효율 계산
    total_input_tokens = total_uncached_input + total_cache_creation + total_cache_reads
    cache_efficiency = (
        (total_cache_reads / total_input_tokens * 100) if total_input_tokens > 0 else 0
    )

    # 요약 표시
    print("📊 Usage Summary:")
    print(f"Uncached input tokens: {total_uncached_input:,}")
    print(f"Output tokens: {total_output:,}")
    print(f"Cache creation: {total_cache_creation:,}")
    print(f"Cache reads: {total_cache_reads:,}")
    print(f"Cache efficiency: {cache_efficiency:.1f}%")
    print(f"Web searches: {total_web_searches:,}")

    return daily_data


# 사용 예제
if client:
    usage_response = get_daily_usage(client, days_back=7)
    daily_usage = analyze_usage_data(usage_response)

📊 Usage Summary:
Uncached input tokens: 267,751
Output tokens: 2,848,746
Cache creation: 0
Cache reads: 0
Cache efficiency: 0.0%
Web searches: 0


## 기본 비용 추적

참고: Priority Tier 비용에는 다른 과금 모델이 사용되며 cost endpoint에는 나타나지 않습니다. Usage endpoint에서 Priority Tier 사용량은 추적할 수 있지만 비용은 추적할 수 없습니다.

In [13]:
def get_daily_costs(client, days_back=7):
    """최근 N일의 비용 데이터를 가져옵니다."""
    end_time = datetime.combine(datetime.utcnow(), time.min)
    start_time = end_time - timedelta(days=days_back)

    params = {
        "starting_at": start_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "ending_at": end_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "bucket_width": "1d",  # Cost API는 1d만 지원
        "limit": min(days_back, 31),  # 요청당 최대 31일
    }

    return client._make_request("cost_report", params)


def analyze_cost_data(response):
    """비용 데이터를 처리하고 표시합니다."""
    if not response or not response.get("data"):
        print("No cost data found.")
        return

    total_cost_minor_units = 0
    daily_costs = []

    for bucket in response["data"]:
        date = bucket["starting_at"][:10]

        # 이 bucket의 모든 비용 합산
        bucket_cost = 0
        for result in bucket["results"]:
            # 필요한 경우 문자열 금액을 float로 변환
            amount = result.get("amount", 0)
            if isinstance(amount, str):
                try:
                    amount = float(amount)
                except (ValueError, TypeError):
                    amount = 0
            bucket_cost += amount

        daily_costs.append(
            {
                "date": date,
                "cost_minor_units": bucket_cost,
                "cost_usd": bucket_cost / 100,  # Dollar로 변환
            }
        )

        total_cost_minor_units += bucket_cost

    total_cost_usd = total_cost_minor_units / 100

    print("💰 Cost Summary:")
    print(f"Total cost: ${total_cost_usd:.4f}")
    print(f"Average daily cost: ${total_cost_usd / len(daily_costs):.4f}")

    return daily_costs


# 사용 예제
if client:
    cost_response = get_daily_costs(client, days_back=7)
    daily_costs = analyze_cost_data(cost_response)

💰 Cost Summary:
Total cost: $83.7574
Average daily cost: $11.9653


## 그룹화, 필터링 및 Pagination

### 시간 단위 옵션

**Usage API**는 세 가지 시간 단위를 지원합니다:
- `1m`(1분): 고해상도 분석, 요청당 최대 1440 bucket
- `1h`(1시간): 중간 해상도, 요청당 최대 168 bucket  
- `1d`(1일): 일별 분석, 요청당 최대 31 bucket

**Cost API**는 다음을 지원합니다:
- `1d`(1일): 유일한 옵션, 요청당 최대 31 bucket

### 그룹화 및 필터링

In [14]:
def get_usage_by_model(client, days_back=7):
    """Pagination을 자동 처리하며 모델별로 그룹화된 사용량 데이터를 가져옵니다."""
    end_time = datetime.combine(datetime.utcnow(), time.min)
    start_time = end_time - timedelta(days=days_back)

    params = {
        "starting_at": start_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "ending_at": end_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "group_by[]": ["model"],
        "bucket_width": "1d",
    }

    # 모든 데이터 페이지에 걸쳐 집계
    model_usage = {}
    page_count = 0
    max_pages = 10  # 무한 loop 방지를 위한 합리적인 제한

    try:
        next_page = None

        while page_count < max_pages:
            current_params = params.copy()
            if next_page:
                current_params["page"] = next_page

            response = client._make_request("usage_report/messages", current_params)
            page_count += 1

            # 이 페이지 데이터 처리
            for bucket in response.get("data", []):
                for result in bucket.get("results", []):
                    model = result.get("model", "Unknown")
                    uncached = result.get("uncached_input_tokens", 0)
                    output = result.get("output_tokens", 0)
                    cache_creation = result.get("cache_creation", {})
                    cache_creation_tokens = cache_creation.get(
                        "ephemeral_1h_input_tokens", 0
                    ) + cache_creation.get("ephemeral_5m_input_tokens", 0)
                    cache_reads = result.get("cache_read_input_tokens", 0)
                    tokens = uncached + output + cache_creation_tokens + cache_reads

                    if model not in model_usage:
                        model_usage[model] = 0
                    model_usage[model] += tokens

            # 추가 데이터가 있는지 확인
            if not response.get("has_more", False):
                break

            next_page = response.get("next_page")
            if not next_page:
                break

    except Exception as e:
        print(f"❌ Error retrieving usage data: {e}")
        return {}

    # 결과 표시
    print("📊 Usage by Model:")
    if not model_usage:
        print(f"  No usage data found in the last {days_back} days")
        print("  💡 Try increasing the time range or check if you have recent API usage")
    else:
        for model, tokens in sorted(model_usage.items(), key=lambda x: x[1], reverse=True):
            print(f"  {model}: {tokens:,} tokens")

    return model_usage


def filter_usage_example(client):
    """사용량 데이터 필터링 예제."""
    params = {
        "starting_at": (datetime.combine(datetime.utcnow(), time.min) - timedelta(days=7)).strftime(
            "%Y-%m-%dT%H:%M:%SZ"
        ),
        "ending_at": datetime.combine(datetime.utcnow(), time.min).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "models[]": ["claude-sonnet-4-6"],  # 특정 모델로 필터링
        "service_tiers[]": ["standard"],  # Standard tier로 필터링
        "bucket_width": "1d",
    }

    response = client._make_request("usage_report/messages", params)
    print(f"Found {len(response.get('data', []))} days of filtered usage data")
    return response


# 사용 예제
if client:
    model_usage = get_usage_by_model(client, days_back=14)
    filtered_usage = filter_usage_example(client)

📊 Usage by Model:
  claude-3-5-haiku-20241022: 995,781 tokens
  claude-sonnet-4-6: 861,880 tokens
  claude-opus-4-1: 394,646 tokens
  claude-sonnet-4-6: 356,766 tokens
  claude-opus-4-20250514: 308,223 tokens
  claude-opus-4-1: 199,201 tokens
Found 7 days of filtered usage data


### 대규모 Dataset을 위한 Pagination

In [18]:
def fetch_all_usage_data(client, params, max_pages=10):
    """Pagination된 모든 사용량 데이터를 가져옵니다."""
    all_data = []
    page_count = 0
    next_page = None

    print("📥 Fetching paginated data...")

    while page_count < max_pages:
        current_params = params.copy()
        if next_page:
            current_params["page"] = next_page

        try:
            response = client._make_request("usage_report/messages", current_params)

            if not response or not response.get("data"):
                break

            page_data = response["data"]
            all_data.extend(page_data)
            page_count += 1

            print(f"  Page {page_count}: {len(page_data)} time buckets")

            if not response.get("has_more", False):
                print(f"✅ Complete: Retrieved all data in {page_count} pages")
                break

            next_page = response.get("next_page")
            if not next_page:
                break

        except Exception as e:
            print(f"❌ Error on page {page_count + 1}: {e}")
            break

    print(f"📊 Total retrieved: {len(all_data)} time buckets")
    return all_data


def large_dataset_example(client, days_back=3):
    """Pagination으로 대규모 dataset을 처리하는 예제."""
    # 데이터가 있도록 최근 시간 범위 사용
    start_time = datetime.combine(datetime.utcnow(), time.min) - timedelta(days=days_back)
    end_time = datetime.combine(datetime.utcnow(), time.min)

    params = {
        "starting_at": start_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "ending_at": end_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "bucket_width": "1h",  # 더 많은 bucket을 위한 시간별 데이터
        "group_by[]": ["model"],
        "limit": 24,  # 페이지당 하루
    }

    all_buckets = fetch_all_usage_data(client, params, max_pages=5)

    # 대규모 dataset 처리
    if all_buckets:
        total_tokens = sum(
            sum(
                result.get("uncached_input_tokens", 0) + result.get("output_tokens", 0)
                for result in bucket["results"]
            )
            for bucket in all_buckets
        )
        print(f"📈 Total tokens across all data: {total_tokens:,}")

    return all_buckets


# 사용 예제 - 최근 데이터를 찾기 위해 짧은 시간 범위 사용
if client:
    large_dataset = large_dataset_example(client, days_back=3)

📥 Fetching paginated data...
  Page 1: 24 time buckets
  Page 2: 24 time buckets
  Page 3: 24 time buckets
✅ Complete: Retrieved all data in 3 pages
📊 Total retrieved: 72 time buckets
📈 Total tokens across all data: 1,336,287


## 간단한 데이터 내보내기

### 외부 분석을 위한 CSV 내보내기

In [16]:
import csv


def export_usage_to_csv(client, output_file="usage_data.csv", days_back=30):
    """외부 분석을 위해 사용량 데이터를 CSV로 내보냅니다."""

    end_time = datetime.combine(datetime.utcnow(), time.min)
    start_time = end_time - timedelta(days=days_back)

    params = {
        "starting_at": start_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "ending_at": end_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "group_by[]": ["model", "service_tier", "workspace_id"],
        "bucket_width": "1d",
    }

    try:
        # 페이지에 걸친 모든 데이터 수집
        rows = []
        page_count = 0
        max_pages = 20  # 내보내기를 위해 더 많은 페이지 허용
        next_page = None

        while page_count < max_pages:
            current_params = params.copy()
            if next_page:
                current_params["page"] = next_page

            response = client._make_request("usage_report/messages", current_params)
            page_count += 1

            # 이 페이지 데이터 처리
            for bucket in response.get("data", []):
                date = bucket["starting_at"][:10]
                for result in bucket["results"]:
                    rows.append(
                        {
                            "date": date,
                            "model": result.get("model", ""),
                            "service_tier": result.get("service_tier", ""),
                            "workspace_id": result.get("workspace_id", ""),
                            "uncached_input_tokens": result.get("uncached_input_tokens", 0),
                            "output_tokens": result.get("output_tokens", 0),
                            "cache_creation_tokens": (
                                result.get("cache_creation", {}).get("ephemeral_1h_input_tokens", 0)
                                + result.get("cache_creation", {}).get(
                                    "ephemeral_5m_input_tokens", 0
                                )
                            ),
                            "cache_read_tokens": result.get("cache_read_input_tokens", 0),
                            "web_search_requests": result.get("server_tool_use", {}).get(
                                "web_search_requests", 0
                            ),
                        }
                    )

            # 추가 데이터가 있는지 확인
            if not response.get("has_more", False):
                break

            next_page = response.get("next_page")
            if not next_page:
                break

        # CSV 기록
        if rows:
            with open(output_file, "w", newline="") as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=rows[0].keys())
                writer.writeheader()
                writer.writerows(rows)

            print(f"✅ Exported {len(rows)} rows to {output_file}")
        else:
            print(f"No usage data to export for the last {days_back} days")
            print("💡 Try increasing days_back or check if you have recent API usage")

    except Exception as e:
        print(f"❌ Export failed: {e}")


def export_costs_to_csv(client, output_file="cost_data.csv", days_back=30):
    """비용 데이터를 CSV로 내보냅니다."""

    end_time = datetime.combine(datetime.utcnow(), time.min)
    start_time = end_time - timedelta(days=days_back)

    params = {
        "starting_at": start_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "ending_at": end_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "group_by[]": ["workspace_id", "description"],
    }

    try:
        # 페이지에 걸친 모든 데이터 수집
        rows = []
        page_count = 0
        max_pages = 20
        next_page = None

        while page_count < max_pages:
            current_params = params.copy()
            if next_page:
                current_params["page"] = next_page

            response = client._make_request("cost_report", current_params)
            page_count += 1

            # 이 페이지 데이터 처리
            for bucket in response.get("data", []):
                date = bucket["starting_at"][:10]
                for result in bucket["results"]:
                    # 문자열 및 숫자 금액 모두 처리
                    amount = result.get("amount", 0)
                    if isinstance(amount, str):
                        try:
                            amount = float(amount)
                        except (ValueError, TypeError):
                            amount = 0

                    rows.append(
                        {
                            "date": date,
                            "workspace_id": result.get(
                                "workspace_id", ""
                            ),  # 기본 workspace의 경우 null
                            "description": result.get("description", ""),
                            "currency": result.get("currency", "USD"),
                            "amount_usd": amount / 100,
                        }
                    )

            # 추가 데이터가 있는지 확인
            if not response.get("has_more", False):
                break

            next_page = response.get("next_page")
            if not next_page:
                break

        if rows:
            with open(output_file, "w", newline="") as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=rows[0].keys())
                writer.writeheader()
                writer.writerows(rows)

            print(f"✅ Exported {len(rows)} cost records to {output_file}")
        else:
            print(f"No cost data to export for the last {days_back} days")
            print("💡 Try increasing days_back or check if you have recent API usage")

    except Exception as e:
        print(f"❌ Cost export failed: {e}")


# 사용 예제
if client:
    export_usage_to_csv(client, "my_usage_data.csv", days_back=14)
    export_costs_to_csv(client, "my_cost_data.csv", days_back=14)

✅ Exported 36 rows to my_usage_data.csv
✅ Exported 72 cost records to my_cost_data.csv


## 마무리

이 cookbook은 Usage & Cost Admin API를 사용하는 핵심 패턴을 다룹니다:

- 사용량 및 비용 데이터의 **기본 조회**
- 상세 분석을 위한 **그룹화 및 필터링**  
- 대규모 dataset을 위한 **pagination**
- 분류를 위한 **비용 설명 parsing**
- 문제를 피하기 위한 **일반적인 주의 사항**
- 외부 도구를 위한 **간단한 CSV 내보내기**

### 다음 단계

- 최신 field 정의는 [공식 API 문서](https://docs.claude.com) 확인
- 먼저 짧은 날짜 범위로 통합 테스트
- 사용 사례에 필요한 데이터 보존 요건 고려
- 분석을 개선할 수 있는 새로운 API 기능 모니터링

### 중요 사항

- API가 성숙함에 따라 field 이름과 사용 가능한 옵션이 바뀔 수 있음
- Production code에서는 알 수 없는 값을 항상 적절히 처리
- API는 실시간 모니터링이 아닌 과거 데이터 분석을 위해 설계됨
- Priority Tier 비용에는 다른 과금 모델이 사용되며 cost endpoint에 나타나지 않음

즐겁게 분석하세요! 📊